# SpeechT5 + WavLM Fine-Tuning

This notebook fine-tunes **SpeechT5** (`microsoft/speecht5_vc`) to accept **WavLM hidden states**
as encoder input instead of raw audio, enabling a WavLM-based feature-space speech translation pipeline.

## Architecture
```
EN raw audio  ──► [WavLM Encoder (frozen)]  ──► EN hidden states (Seq, 768)
                                                        │
                                 [SpeechT5 Transformer (fine-tuned)] ──► DE mel spectrogram
                                                        │
                                             [HiFi-GAN Vocoder]  ──► DE audio
```

## Required Pre-Step
The preprocessed WavLM dataset must exist at:
```
datasets/processed_wavlm_en_de_v1/
    en/   ← WavLM hidden states (Seq, 768)
    de/   ← WavLM hidden states (Seq, 768)
```
Run `preprocess_wavlm.py` if the dataset is not yet generated.

> **Note on the target representation:**  
> The dataset stores WavLM features for both sides. The `SpeechT5WavLMDataset`
> class in `model.py` handles the fallback to mel-spectrogram extraction if the
> target is raw audio. The fine-tuning loss is computed against SpeechT5's
> mel-spectrogram decoder output — this is what teaches the model to bridge
> the WavLM representation into something the trained decoder can process.


## 1. Setup & Imports

In [1]:
import os
import sys

# Add project root to Python path so that dataset_loader and encoders are importable.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

from model import SpeechT5WavLM
import dataset_loader

print(f"Project root: {project_root}")
print(f"Datasets directory: {dataset_loader.DATASETS_DIR}")

/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model
Datasets directory: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets


## 2. Configuration

Edit the constants below to control training behaviour.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Path to the preprocessed WavLM dataset produced by preprocess_wavlm.py.
# ─────────────────────────────────────────────────────────────────────────────
PREPROCESSED_PATH = os.path.join(dataset_loader.DATASETS_DIR, "processed_wavlm_en_de_v1")

# ─────────────────────────────────────────────────────────────────────────────
# Training hyper-parameters
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS        = 5        # Total training epochs
BATCH_SIZE    = 1        # Keep small — WavLM sequences are variable length
LEARNING_RATE = 5e-6     # Low LR for stable fine-tuning of pre-trained weights

# ─────────────────────────────────────────────────────────────────────────────
# Output checkpoint directory
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "speecht5_wavlm_en_de_v1"

print(f"Preprocessed data: {PREPROCESSED_PATH}")
print(f"Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"Checkpoint will be saved to: {CHECKPOINT_DIR}")

Preprocessed data: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v1
Epochs: 5  |  Batch size: 1  |  LR: 5e-06
Checkpoint will be saved to: speecht5_wavlm_en_de_v1


## 3. Verify Preprocessed Dataset

Confirm the dataset exists and inspect a few sample feature shapes before starting training.

In [3]:
from datasets import load_from_disk
import numpy as np

assert os.path.isdir(PREPROCESSED_PATH), (
    f"Preprocessed dataset not found at '{PREPROCESSED_PATH}'.\n"
    "Run preprocess_wavlm.py first."
)

# Detect language subdirectories automatically (matches model.py behaviour)
lang_dirs = sorted([
    d for d in os.listdir(PREPROCESSED_PATH)
    if os.path.isdir(os.path.join(PREPROCESSED_PATH, d))
])
assert len(lang_dirs) >= 2, f"Expected at least 2 language dirs, found: {lang_dirs}"

source_lang, target_lang = lang_dirs[0], lang_dirs[1]
print(f"Source language: {source_lang}")
print(f"Target language: {target_lang}")

src_ds = load_from_disk(os.path.join(PREPROCESSED_PATH, source_lang))
tgt_ds = load_from_disk(os.path.join(PREPROCESSED_PATH, target_lang))

print(f"\nTotal pairs: {len(src_ds)}")
print(f"Source columns: {src_ds.column_names}")
print(f"Target columns: {tgt_ds.column_names}")

print("\nSample feature shapes (first 5 pairs):")
for i in range(min(5, len(src_ds))):
    sf = np.array(src_ds[i]['audio'])
    tf = np.array(tgt_ds[i]['audio'])
    print(f"  [{i}] EN shape={sf.shape}  mean={sf.mean():.4f}  std={sf.std():.4f} | "
          f"DE shape={tf.shape}  mean={tf.mean():.4f}  std={tf.std():.4f}")

Source language: de
Target language: en

Total pairs: 13438
Source columns: ['id', 'audio', 'language', 'gender', 'transcription']
Target columns: ['id', 'audio', 'language', 'gender', 'transcription']

Sample feature shapes (first 5 pairs):
  [0] EN shape=(206, 768)  mean=0.0039  std=0.2030 | DE shape=(212, 768)  mean=0.0042  std=0.2031
  [1] EN shape=(140, 768)  mean=0.0050  std=0.2155 | DE shape=(145, 768)  mean=0.0037  std=0.1970
  [2] EN shape=(217, 768)  mean=0.0048  std=0.2111 | DE shape=(217, 768)  mean=0.0047  std=0.2110
  [3] EN shape=(120, 768)  mean=0.0048  std=0.2133 | DE shape=(102, 768)  mean=0.0048  std=0.2008
  [4] EN shape=(278, 768)  mean=0.0044  std=0.2115 | DE shape=(192, 768)  mean=0.0039  std=0.2018


## 4. Initialise the Model

`SpeechT5WavLM` loads:
- **SpeechT5** (`microsoft/speecht5_vc`) — the transformer to fine-tune
- **HiFi-GAN** (`microsoft/speecht5_hifigan`) — the vocoder (frozen, used at inference)

WavLM itself is **not** loaded here; it was already used during preprocessing to produce the
hidden states stored in the dataset. It is only needed again at inference time
(handled by `run_inference`).

In [4]:
model = SpeechT5WavLM()
print(f"\nModel loaded on device: {model.device}")

Loading SpeechT5WavLM components (WavLM=microsoft/wavlm-base-plus)...
Using device: cuda

Model loaded on device: cuda


## 5. Extract Target Speaker Embedding

The SpeechT5 decoder needs an **x-vector speaker embedding** to condition its output voice.

`get_speaker_embedding` streams one sample from Google FLEURS for the target language and
computes the x-vector using the SpeechBrain classifier. The embedding is saved alongside
the model checkpoint.

In [5]:
model.get_speaker_embedding(target_lang=target_lang)

Initializing X-Vector classifier for embedding extraction...


/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


Extracting target speaker embedding...
Embedding extracted successfully.


## 6. Fine-Tune

Training will:
1. Load the preprocessed WavLM dataset.
2. Freeze SpeechT5's CNN feature encoder (only transformer layers are trained).
3. Feed WavLM hidden states directly into SpeechT5's transformer encoder.
4. Compute the spectrogram loss against the decoder's mel output.
5. Save a checkpoint every 5 epochs. On `KeyboardInterrupt` progress is saved
   automatically to `speecht5_wavlm_interrupted`.

In [6]:
model.fine_tune(
    preprocessed_path=PREPROCESSED_PATH,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

Starting WavLM+SpeechT5 fine-tuning.
Loading preprocessed data from /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/processed_wavlm_en_de_v1...
Freezing CNN Feature Encoder.


Epoch 1/5:  79%|███████▊  | 10563/13438 [31:45<08:38,  5.54it/s, loss=0.2557] 
Traceback (most recent call last):
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/models/SpeechT5WavLM/model.py", line 362, in fine_tune
    loss = criterion(
           ^^^^^^^^^^
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Pr


An error occurred: CUDA out of memory. Tried to allocate 84.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 88.88 MiB is free. Including non-PyTorch memory, this process has 3.57 GiB memory in use. Of the allocated memory 3.32 GiB is allocated by PyTorch, and 144.08 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Saving current progress to 'speecht5_wavlm_error_mid_train'...
Progress saved. Exiting safely.


## 7. Save the Final Checkpoint

In [7]:
model.save(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")

Model saved to: speecht5_wavlm_en_de_v1


## 8. Quick Inference Test

Load a raw EN clip from the raw dataset and run it through the full pipeline
(WavLM → fine-tuned SpeechT5 transformer → HiFi-GAN) to hear if the model is learning.

In [8]:
from IPython.display import Audio, display
import numpy as np

model.load(CHECKPOINT_DIR)
# Load a raw EN sample from the seamless_align dataset for inference.
print("Loading a raw EN sample for inference test...")
raw = dataset_loader.load_data(
    lang=[source_lang],
    split="train",
    dataset=["fleurs"],
    num_samples=100,
)
sample = raw[source_lang][0]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']

print(f"Input audio: {len(audio_array)/sr:.2f}s @ {sr}Hz")
print("Original EN audio:")
display(Audio(data=audio_array, rate=sr))

# Run the full pipeline: raw audio → WavLM → fine-tuned SpeechT5 → HiFi-GAN → audio
print("\nRunning fine-tuned inference...")
result = model.run_inference(
    audio_array=audio_array,
    sampling_rate=sr,
    threshold=0.5,
    minlenratio=0.0,
    maxlenratio=2.0,
)

out_audio = result['audio']['array']
out_sr    = result['audio']['sampling_rate']

print(f"Output audio: {len(out_audio)/out_sr:.2f}s @ {out_sr}Hz")
print("\nGenerated DE audio (after fine-tuning):")
display(Audio(data=out_audio, rate=out_sr))

Loading a raw EN sample for inference test...
Loading google/fleurs (de_de) from local storage: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/fleurs/de_de/train...
Slicing loaded dataset (0:100)...
Validating de (checking audio & uniqueness)...
  100 valid unique IDs found.
Final Count: 100 common valid samples.
Input audio: 9.90s @ 16000Hz
Original EN audio:



Running fine-tuned inference...
Loading WavLM (microsoft/wavlm-base-plus) extractor for inference...
Output audio: 15.81s @ 16000Hz

Generated DE audio (after fine-tuning):


## Bonus: Resume Training from a Checkpoint

If training was interrupted, load the saved checkpoint and continue.

In [9]:
# ── Uncomment to resume from a saved checkpoint ──────────────────────────────
# RESUME_CHECKPOINT = "speecht5_wavlm_interrupted"   # or "checkpoint_epoch_5", etc.
# model_resume = SpeechT5WavLM()
# model_resume.load(RESUME_CHECKPOINT)
# model_resume.fine_tune(
#     preprocessed_path=PREPROCESSED_PATH,
#     epochs=EPOCHS,
#     learning_rate=LEARNING_RATE,
#     batch_size=BATCH_SIZE,
# )